# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Metadata is provided as an mlcroissant.Metadata object
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Version: {metadata.version}")
print(f"Identifier: {metadata.identifier}")
print(f"Description: {metadata.description}\n")
print("Keywords:", metadata.keywords)

print("\n-- Dataset Overview --")
print(f"Record sets: {len(metadata.record_sets) if hasattr(metadata, 'record_sets') else 'Not found'}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore record sets and their fields by @id

record_sets = getattr(metadata, 'record_sets', [])  # list of mlcroissant.RecordSet
if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}")
        print(f"  Name: {rs.name}")
        fields = getattr(rs, 'fields', [])
        columns = getattr(rs, 'columns', [])
        print(f"  Fields ({len(fields)}):", [f.id for f in fields])
        print(f"  Columns ({len(columns)}):", [c.id for c in columns])
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# If the dataset has multiple record sets, list their @ids. Choose the main one for demo purposes.

dataframes = {}

# Get all record set @ids
record_sets = getattr(metadata, 'record_sets', [])
record_set_ids = [rs.id for rs in record_sets]
print(f"Available RecordSet @ids: {record_set_ids}\n")

# For this notebook, we'll usually have only 1 main RecordSet. Use index 0 for demo.
if record_set_ids:
    main_record_set_id = record_set_ids[0]

    # Show all fields in this record set:
    rs = [rs for rs in record_sets if rs.id == main_record_set_id][0]
    field_ids = [field.id for field in getattr(rs, 'fields', [])]
    print(f"Fields for RecordSet {main_record_set_id}: {field_ids}\n")

    # Load records into a pandas DataFrame
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dataframes[main_record_set_id] = df

    print(f"Columns in DataFrame: {df.columns.tolist()}")
    display(df.head())
else:
    print("No record sets found; cannot proceed with extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. For demonstration, use GAD-7, PHQ-9, or PSQ scores if available.

In [ ]:
import numpy as np

# Typical score fields to explore
possible_score_fields = [col for col in df.columns if any(k in col.lower() for k in ['gad7', 'phq9', 'psq', 'score'])]
print("Possible numeric (score) fields:", possible_score_fields)

if possible_score_fields:
    numeric_field = possible_score_fields[0]  # e.g. 'phq9_score' or original column @id
    threshold = 10  # domain-specific threshold for moderate severity, e.g., for PHQ-9

    # Filter for severe cases
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df[[numeric_field]].head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a demographic field, e.g. 'gender' or 'age_group', if present
    # Use field 'gender' or similar
    possible_groups = [col for col in df.columns if 'gender' in col.lower() or 'group' in col.lower() or 'education' in col.lower()]
    if possible_groups:
        group_field = possible_groups[0]
        print(f"Grouping data by '{group_field}'...")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().rename({numeric_field: f'mean_{numeric_field}'}, axis=1)
        display(grouped_df)
    else:
        print("No suitable group field found for grouping.")
else:
    print("No suitable numeric field identified for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the chosen numeric field
if possible_score_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field if available
    if possible_groups:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was successfully loaded using the Croissant schema and `mlcroissant`.
- The main record set contains demographic and mental health assessment data (GAD-7, PHQ-9, PSQ, etc.).
- You can filter for high-risk groups (e.g., PHQ-9 > 10), normalize fields, and group by demographics (e.g., gender).
- The data is ready for further analysis, such as detection of survey patterns and mental health risk categories.

Continue with domain-specific analyses and modeling as required.